# Full Preprocessing Pipeline (Verbose & Multi-Task)

This notebook implements the complete preprocessing strategy defined in `PREPROCESSING_GUIDE.md` with **live logging**.

## Objectives:
1. **Initial Validation**: Drop duplicates and nulls from the raw dataset.
2. **Structural Cleaning**: Remove GIFs, stickers, and tags. Drop tag-only comments.
3. **Text Normalization**: Map punctuation intensity to `[INTENSE]`.
4. **Multi-Task Emoji Mapping**: Map emojis to tokens based on the selected task.
5. **Language Filtering**: Keep only Arabic and Latin scripts.
6. **Empty Removal**: Explicitly drop rows that become empty after cleaning.
7. **Verbose Logs**: See exactly what changed for each comment.

In [15]:
import pandas as pd
import re
import emoji
import random
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 42
INPUT_FILE = 'dataset.csv'
SAMPLE_SIZE = 200 # Adjust for verbose speed

## 1. Initial Data Cleaning
Before we select a task, we perform a global deduplication and null-check to ensure the base dataset is clean.

In [16]:
print(f"Loading {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)
initial_count = len(df)

# 1. Drop absolute empty comments
df = df.dropna(subset=['comment_text'])
after_nulls = len(df)

# 2. Drop duplicates in comment_text
df = df.drop_duplicates(subset=['comment_text'])
after_dedup = len(df)

print(f"--- Initial Cleaning Results ---")
print(f"Raw rows: {initial_count}")
print(f"Dropped {initial_count - after_nulls} null rows.")
print(f"Dropped {after_nulls - after_dedup} duplicate rows.")
print(f"Total unique usable rows: {len(df)}")

Loading dataset.csv...
--- Initial Cleaning Results ---
Raw rows: 43788
Dropped 82 null rows.
Dropped 9252 duplicate rows.
Total unique usable rows: 34454


## 2. Select Preprocessing Task

In [17]:
print("Select Preprocessing Task:")
print("1. Sentiment Analysis")
print("2. Intent Classification")
print("3. Topic Classification")

choice = input("Enter choice (1/2/3): ")
modes = {"1": "SENTIMENT", "2": "INTENT", "3": "TOPIC"}
MODE = modes.get(choice, "SENTIMENT")
print(f"\n>>> Active Mode: {MODE}")

Select Preprocessing Task:
1. Sentiment Analysis
2. Intent Classification
3. Topic Classification

>>> Active Mode: TOPIC


## 3. Core Functions

In [ ]:
def log_transform(step_name, original, result):
    if str(original).strip() != str(result).strip():
        print(f"  [{step_name}] ")
        print(f"    - Before: {original}")
        print(f"    - After : {result}")

def full_pipeline(text, mode, verbose=True):
    if not isinstance(text, str): return ""
    current = text.strip()
    if verbose: print(f"\n--- Processing: '{current}'")

    # 1. Structural
    temp = re.sub(r"\[GIF\]|\[Sticker\]", "", current)
    if verbose: log_transform("Structural", current, temp)
    current = temp

    # 2. Reply Prefixes
    temp = re.sub(r"^(@[\w.]+[: ]*|Replying to @[\w.]+[: ]*)", "", current)
    if verbose: log_transform("Prefix Strip", current, temp)
    current = temp

    # 3. Tags & Empty Check
    no_tags = re.sub(r"@\w+", "", current).strip()
    if not no_tags:
        if verbose: print("  [DROP] Reason: Tags only comment")
        return ""
    if verbose: log_transform("Internal Tags", current, no_tags)
    current = no_tags

    # 4. Punctuation Intensity
    temp = re.sub(r"[!]{2,}|[?]{2,}|[.]{3,}", " [INTENSE] ", current).strip()
    if verbose: log_transform("Intensity", current, temp)
    current = temp

    # 5. Emoji Mapping
    emojis_found = [e['emoji'] for e in emoji.emoji_list(current)]
    tokens = []
    
    if mode == "SENTIMENT":
        pos = ['❤️', '🥰', '😍', '🔥', '😋', '😂', '👏', '💯', '👍', '😁', '🤩', '😊', '🥳', '💪', '🤲', '🌹', '💐', '💎', '🇩🇿']
        neg = ['🤮', '😡', '👎', '💔', '💀', '💸', '😭', '😢', '😒', '😑', '😱']
        if any(e in pos for e in emojis_found): tokens.append("[POS]")
        if any(e in neg for e in emojis_found): tokens.append("[NEG]")
        
    elif mode == "INTENT":
        appr = ['❤️', '🥰', '😂', '👏', '🤲', '🌹', '💐']
        comp = ['🤮', '😡', '👎', '💔', '💀', '💸', '😒', '😑']
        inq = ['❓', '❔', '🤔', '🧐', '👀', '📍', '📞', '🕒']
        recom = ['👌', '🔝', '🌟', '✨', '✅', '🥇', '👑']
        if any(e in appr for e in emojis_found): tokens.append("[APPRECIATION]")
        if any(e in comp for e in emojis_found): tokens.append("[COMPLAINT]")
        if any(e in inq for e in emojis_found): tokens.append("[INQUIRY]")
        if any(e in recom for e in emojis_found): tokens.append("[RECOMMENDATION]")
        # if not tokens and emojis_found: tokens.append("[OUT_OF_SCOPE]")

    elif mode == "TOPIC":
        bouffe = ['🥘', '🍔', '🍕', '🥙', '🥗', '🍦', '😋', '🤤', '🍜', '🍣', '🥩']
        price = ['💸', '💰', '💳', '💶', '💵']
        treat = ['🧑‍🍳', '👨‍🍳', '👋', '🤝', '🫂']
        srv = ['🕒', '⏳', '🛵', '🍴', '🍽️']
        endroit = ['📍', '🧼', '🧹', '📸', '🤳', '✨', '🌟', '🏝']
        delivery = ['🛵', '🚚', '📦']
        if any(e in bouffe for e in emojis_found): tokens.append("[BOUFFE]")
        if any(e in price for e in emojis_found): tokens.append("[PRICE]")
        if any(e in treat for e in emojis_found): tokens.append("[TREATMENT]")
        if any(e in srv for e in emojis_found): tokens.append("[SERVICE]")
        if any(e in endroit for e in emojis_found): tokens.append("[ENDROIT]")
        if any(e in delivery for e in emojis_found): tokens.append("[DELIVERY]")
        # if not tokens and emojis_found: tokens.append("[UNKNOWN]")

    temp = emoji.replace_emoji(current, replace="")
    final = (temp + " " + " ".join(tokens)).strip()
    if verbose: log_transform("Emoji Mapping", current, final)
    
    return final

def is_target_language(text):
    if len(text.strip()) < 2: return False
    try:
        lang = detect(text)
        return lang in ['ar', 'fr', 'en']
    except:
        return False

## 4. Execute Pipeline

In [20]:
# Filter a sample for verbose tracing from our clean set
sample_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
sample_df['raw_text'] = sample_df['comment_text']

print(f"Executing Full Pipeline for {MODE} mode...\n")
sample_df['comment_text'] = sample_df['comment_text'].apply(lambda x: full_pipeline(x, MODE, verbose=True))

# 6. Drop empty/null comments resulting from cleaning
print("\n--- Cleaning Finished. Removing empty results... ---")
count_before = len(sample_df)
sample_df = sample_df[sample_df['comment_text'].str.strip() != ""]
sample_df = sample_df.dropna(subset=['comment_text'])
print(f"Dropped {count_before - len(sample_df)} empty/unusable rows.")

# 7. Filter by Language (Arabic/Latin)
print("Enforcing script rules...")
count_before = len(sample_df)
sample_df = sample_df[sample_df['comment_text'].apply(is_target_language)]
print(f"Dropped {count_before - len(sample_df)} rows due to language filtering.")

# 8. Final cleanup: dedup and shuffle
sample_df = sample_df.drop_duplicates(subset=['comment_text'])
sample_df = sample_df.sample(frac=1, random_state=42).reset_index(drop=True)
sample_df.insert(0, 'final_id', range(1, len(sample_df) + 1))

output_name = f'processed_sample_{MODE.lower()}.csv'
sample_df.to_csv(output_name, index=False)
print(f"\nSUCCESS! Saved {len(sample_df)} rows to {output_name}")

Executing Full Pipeline for TOPIC mode...


--- Processing: 'Madabikom w ila bghito rabhouna ana w sahbi@Le R🫀 wlh mankousoumo bzf li ta3touhalna nafarho bih mohim narbho nhaso b goût ta3 rabh🤣'
  [Internal Tags] 
    - Before: Madabikom w ila bghito rabhouna ana w sahbi@Le R🫀 wlh mankousoumo bzf li ta3touhalna nafarho bih mohim narbho nhaso b goût ta3 rabh🤣
    - After : Madabikom w ila bghito rabhouna ana w sahbi R🫀 wlh mankousoumo bzf li ta3touhalna nafarho bih mohim narbho nhaso b goût ta3 rabh🤣
  [Emoji Mapping] 
    - Before: Madabikom w ila bghito rabhouna ana w sahbi R🫀 wlh mankousoumo bzf li ta3touhalna nafarho bih mohim narbho nhaso b goût ta3 rabh🤣
    - After : Madabikom w ila bghito rabhouna ana w sahbi R wlh mankousoumo bzf li ta3touhalna nafarho bih mohim narbho nhaso b goût ta3 rabh

--- Processing: '@badisnouar your next birthday cake for SURE 🤣🤣'
  [Prefix Strip] 
    - Before: @badisnouar your next birthday cake for SURE 🤣🤣
    - After : your next birthday cake for S

In [21]:
pd.set_option('display.max_colwidth', None)
display(sample_df[['raw_text', 'comment_text']].head(20))

,raw_text,comment_text
0,الزوالي يقدر يدخل هنا باش يشوف هذه المتعة و لا قصدي على ر,الزوالي يقدر يدخل هنا باش يشوف هذه المتعة و لا قصدي على ر
1,نديرو فالطاجين😁,نديرو فالطاجين
2,Vous cherchez quelqu’un travailler avec vous ?,Vous cherchez quelqu’un travailler avec vous ?
3,win exct f sabah,win exct f sabah
4,والله غير حجة مليحة,والله غير حجة مليحة
5,top service 🫶,top service
6,j'arrive 😅 vous êtes ou ??,j'arrive vous êtes ou [INTENSE]
7,قول صحن زيت مش صحن ملوخية 🤢,قول صحن زيت مش صحن ملوخية
8,merci cousin les hommes chez les oranée bientôt à oran 🤲🤲🤲,merci cousin les hommes chez les oranée bientôt à oran
9,واين جاي المحل,واين جاي المحل
